# Premier League Predictor

We attempt to use an XG-Boost model to predict the possible outcomes of premier league games. In this notebook we take the approch of using of treating the home and away scores as regression problems separately. We use data from the website https://www.football-data.co.uk/englandm.php

## Imports and obtaining the data

In [3]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error
# import shap
import matplotlib.pyplot as plt

import requests
from bs4 import BeautifulSoup
import re

from itertools import product
import time


from datetime import datetime

import seaborn as sns
from sklearn.inspection import permutation_importance

The files on my computer obtained from the website (only files for more recent seasons have the kick-off time in them, so we disregard older seasons - we might return to this later)

In [4]:
df2526 = pd.read_csv('25-26.csv')

df2425 = pd.read_csv('24-25.csv')

df2324 = pd.read_csv('23-24.csv')

df2223 = pd.read_csv('22-23.csv')

df2122 = pd.read_csv('21-22.csv')

df2021 = pd.read_csv('20-21.csv')

df1920 = pd.read_csv('19-20.csv')

In [5]:
df1 =  df2526[['Date', 'Time', 'HomeTeam', 'AwayTeam', 'FTHG', 'FTAG']]

df2 =  df2425[['Date', 'Time', 'HomeTeam', 'AwayTeam', 'FTHG', 'FTAG']]

df3 =  df2324[['Date', 'Time', 'HomeTeam', 'AwayTeam', 'FTHG', 'FTAG']]

df4 =  df2223[['Date', 'Time', 'HomeTeam', 'AwayTeam', 'FTHG', 'FTAG']]

df5 =  df2122[['Date', 'Time', 'HomeTeam', 'AwayTeam', 'FTHG', 'FTAG']]

df6 =  df2021[['Date', 'Time', 'HomeTeam', 'AwayTeam', 'FTHG', 'FTAG']]

df7 =  df1920[['Date', 'Time', 'HomeTeam', 'AwayTeam', 'FTHG', 'FTAG']]

df1.shape

(380, 6)

In [6]:
dfnew = pd.concat([df7,df6, df5, df4, df3, df2, df1])
dfnew.shape

(2660, 6)

We proceed to give some functions that will be useful in calculating league positions

In [7]:
def calculate_league_positions(df, final_standings):
    """
    Calculate league positions for home and away teams at the time of each match.
    
    Parameters:
    df: DataFrame with match data
    final_standings: List of team names in final league order
    
    Returns:
    DataFrame with added columns: HomeTeam_Position and AwayTeam_Position
    """
    
    # Create a copy to avoid modifying original
    df = df.copy()
    
    # Convert Date to datetime if it's not already
    df['Date'] = pd.to_datetime(df['Date'], dayfirst=True)
    
    # Extract season from date
    # Football season: Aug-May, so if month >= 8, season is year/year+1, else (year-1)/year
    df['Season'] = df['Date'].apply(lambda x: f"{x.year}-{str(x.year+1)[-2:]}" if x.month >= 8 else f"{x.year-1}-{str(x.year)[-2:]}")
    
    # Determine which columns to use for goals
    if 'FTHG' in df.columns:
        home_goals_col = 'FTHG'
        away_goals_col = 'FTAG'
    elif 'HG' in df.columns:
        home_goals_col = 'HG'
        away_goals_col = 'AG'
    else:
        raise ValueError("Cannot find home/away goals columns (expected FTHG/HG and FTAG/AG)")
    
    # Sort by date to process matches chronologically
    df = df.sort_values('Date').reset_index(drop=True)
    
    # Initialize columns for positions
    df['HomeTeam_Position'] = np.nan
    df['AwayTeam_Position'] = np.nan
    
    # Process each season separately
    for season in df['Season'].unique():
        season_mask = df['Season'] == season
        season_indices = df[season_mask].index
        
        # Initialize league table for this season
        league_table = {}
        teams_in_season = set(df[season_mask]['HomeTeam'].unique()) | set(df[season_mask]['AwayTeam'].unique())
        
        for team in teams_in_season:
            league_table[team] = {'points': 0, 'gd': 0, 'gf': 0, 'played': 0}
        
        # Process each match in chronological order
        for idx in season_indices:
            home_team = df.loc[idx, 'HomeTeam']
            away_team = df.loc[idx, 'AwayTeam']
            
            # Record positions BEFORE this match
            positions = get_league_positions(league_table, final_standings)
            df.loc[idx, 'HomeTeam_Position'] = positions.get(home_team, np.nan)
            df.loc[idx, 'AwayTeam_Position'] = positions.get(away_team, np.nan)
            
            # Get match result from goals
            home_goals = df.loc[idx, home_goals_col]
            away_goals = df.loc[idx, away_goals_col]
            
            # Update goals
            league_table[home_team]['gf'] += home_goals
            league_table[away_team]['gf'] += away_goals
            league_table[home_team]['gd'] += (home_goals - away_goals)
            league_table[away_team]['gd'] += (away_goals - home_goals)
            league_table[home_team]['played'] += 1
            league_table[away_team]['played'] += 1
            
            # Update points based on goals (deduce result)
            if home_goals > away_goals:  # Home win
                league_table[home_team]['points'] += 3
            elif away_goals > home_goals:  # Away win
                league_table[away_team]['points'] += 3
            else:  # Draw
                league_table[home_team]['points'] += 1
                league_table[away_team]['points'] += 1
    
    return df


def get_league_positions(league_table, final_standings):
    """
    Calculate current league positions based on points, goal difference, and goals scored.
    Uses final standings to break ties for teams with identical records.
    
    Parameters:
    league_table: Dictionary with team stats
    final_standings: List of teams in final league order (for tiebreaking)
    
    Returns:
    Dictionary mapping team names to positions
    """
    # Create list of (team, points, gd, gf, played, final_position)
    standings = []
    for team, stats in league_table.items():
        try:
            final_pos = final_standings.index(team)
        except ValueError:
            # Team not in final standings (e.g., relegated in previous season)
            final_pos = 999  # Put at end
        
        standings.append((
            team,
            stats['points'],
            stats['gd'],
            stats['gf'],
            stats['played'],
            final_pos
        ))
    
    # Sort by: points (desc), goal difference (desc), goals for (desc), final position (asc)
    standings.sort(key=lambda x: (-x[1], -x[2], -x[3], x[5]))
    
    # Create position mapping
    positions = {}
    for pos, (team, _, _, _, _, _) in enumerate(standings, 1):
        positions[team] = pos
    
    return positions

In [8]:
def get_premier_league_standings():
    """
    Scrape current Premier League standings from BBC Sport.
    
    Returns:
    List of team names in order from 1st to 20th place
    """
    
    url = "https://www.bbc.co.uk/sport/football/premier-league/table"
    
    print("Fetching Premier League standings from BBC Sport...")
    
    # Add headers to mimic a browser request
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
    }
    
    try:
        # Fetch the page
        response = requests.get(url, headers=headers)
        response.raise_for_status()
        
        # Parse HTML
        soup = BeautifulSoup(response.content, 'html.parser')
        
        # Find the Premier League table
        # BBC uses various class names, so we'll search flexibly
        tables = soup.find_all('table', class_=re.compile('.*table.*', re.IGNORECASE))
        
        if not tables:
            # Try alternative structure
            tables = soup.find_all('div', class_=re.compile('.*league.*table.*', re.IGNORECASE))
        
        final_standings = []
        
        # Look for team names in the table
        # BBC typically uses team names in links or specific span/div elements
        team_elements = soup.find_all('span', class_=re.compile('.*team.*name.*', re.IGNORECASE))
        
        if not team_elements:
            # Try finding by links
            team_elements = soup.find_all('a', class_=re.compile('.*team.*', re.IGNORECASE))
        
        if not team_elements:
            # Try finding table rows
            rows = soup.find_all('tr')
            for row in rows:
                team_cell = row.find('td', class_=re.compile('.*team.*', re.IGNORECASE))
                if team_cell:
                    team_name = team_cell.get_text(strip=True)
                    if team_name and len(team_name) > 2:  # Filter out empty or very short strings
                        final_standings.append(normalize_team_name(team_name))
        else:
            for element in team_elements:
                team_name = element.get_text(strip=True)
                if team_name and len(team_name) > 2:
                    final_standings.append(normalize_team_name(team_name))
        
        # Remove duplicates while preserving order
        seen = set()
        final_standings = [x for x in final_standings if not (x in seen or seen.add(x))]
        
        # Limit to 20 teams (Premier League size)
        final_standings = final_standings[:20]
        
        if len(final_standings) < 20:
            print(f"Warning: Only found {len(final_standings)} teams")
        
        return final_standings
        
    except Exception as e:
        print(f"Error fetching standings: {e}")
        print("Returning example standings instead...")
        # Return a default list if scraping fails
        return get_example_standings()


def normalize_team_name(name):
    """
    Normalize team names to match the format used in the dataset.
    
    Parameters:
    name: Raw team name from BBC
    
    Returns:
    Normalized team name
    """
    
    # Remove extra whitespace
    name = ' '.join(name.split())
    
    # Common name mappings from BBC to dataset format
    name_mappings = {
        'Manchester United': 'Man United',
        'Manchester City': 'Man City',
        'Tottenham Hotspur': 'Tottenham',
        'Brighton and Hove Albion': 'Brighton & Hove Albion',
        'Brighton & Hove Albion': 'Brighton & Hove Albion',
        'Nottingham Forest': "Nott'm Forest",
        "Nott'm Forest": "Nott'm Forest",
        'Wolverhampton Wanderers': 'Wolves',
        'West Ham United': 'West Ham',
        'Newcastle United': 'Newcastle',
        'Leicester City': 'Leicester',
        'Aston Villa': 'Aston Villa',
        'Crystal Palace': 'Crystal Palace',
        'Everton': 'Everton',
        'Brentford': 'Brentford',
        'Fulham': 'Fulham',
        'Chelsea': 'Chelsea',
        'Arsenal': 'Arsenal',
        'Liverpool': 'Liverpool',
        'Bournemouth': 'Bournemouth',
        'AFC Bournemouth': 'Bournemouth',
        'Luton Town': 'Luton',
        'Burnley': 'Burnley',
        'Sheffield United': 'Sheffield United',
        'Leeds United': 'Leeds',
        'Southampton': 'Southampton',
        'Norwich City': 'Norwich',
        'Watford': 'Watford',
        'West Bromwich Albion': 'West Brom',
        'Ipswich Town': 'Ipswich',
        'Sunderland': 'Sunderland'
    }
    
    return name_mappings.get(name, name)


#def get_example_standings():
#    """
#    Return example standings (fallback if scraping fails).
#    """
#    return [
#        'Liverpool', 'Arsenal', 'Chelsea', 'Man City', 'Newcastle',
#        'Bournemouth', "Nott'm Forest", 'Aston Villa', 'Fulham', 'Tottenham',
#        'Brentford', 'Brighton & Hove Albion', 'Man United', 'West Ham', 'Crystal Palace',
#        'Everton', 'Wolves', 'Ipswich', 'Leicester', 'Southampton'
#    ]

In [9]:
# ============================================================================
# USAGE
# ============================================================================

# Get current standings
final_standings = get_premier_league_standings()

print("\n" + "="*70)
print("CURRENT PREMIER LEAGUE STANDINGS")
print("="*70)
for i, team in enumerate(final_standings, 1):
    print(f"{i:2d}. {team}")

print("\n" + "="*70)
print("PYTHON LIST FORMAT")
print("="*70)
print(f"final_standings = {final_standings}")

Fetching Premier League standings from BBC Sport...

CURRENT PREMIER LEAGUE STANDINGS
 1. Bournemouth
 2. Arsenal
 3. Aston Villa
 4. Brentford
 5. Brighton & Hove Albion
 6. Chelsea
 7. Coventry City
 8. Crystal Palace
 9. Everton
10. Fulham
11. Hull City
12. Ipswich
13. Leeds
14. Liverpool
15. Man City
16. Man United
17. Newcastle
18. Nott'm Forest
19. Sunderland
20. Tottenham

PYTHON LIST FORMAT
final_standings = ['Bournemouth', 'Arsenal', 'Aston Villa', 'Brentford', 'Brighton & Hove Albion', 'Chelsea', 'Coventry City', 'Crystal Palace', 'Everton', 'Fulham', 'Hull City', 'Ipswich', 'Leeds', 'Liverpool', 'Man City', 'Man United', 'Newcastle', "Nott'm Forest", 'Sunderland', 'Tottenham']


In [10]:
df_with_positions = calculate_league_positions(dfnew, final_standings)
df_with_positions.head()

,Date,Time,HomeTeam,AwayTeam,FTHG,FTAG,Season,HomeTeam_Position,AwayTeam_Position
0,2019-08-09,20:00,Liverpool,Norwich,4,1,2019-20,7.0,14.0
1,2019-08-10,12:30,West Ham,Man City,0,5,2019-20,13.0,8.0
2,2019-08-10,15:00,Bournemouth,Sheffield United,1,1,2019-20,3.0,18.0
3,2019-08-10,15:00,Burnley,Southampton,3,0,2019-20,15.0,16.0
4,2019-08-10,15:00,Crystal Palace,Everton,0,0,2019-20,9.0,10.0


To establish a baseline, we consider the average home and away scores.

In [11]:
df_with_positions['FTHG'].mean()

1.550751879699248

In [12]:
df_with_positions['FTAG'].mean()

1.3135338345864662

So the basline is a 2-1 home win

In [13]:
df_with_positions[(df_with_positions['FTHG'] == 2) & (df_with_positions['FTAG'] == 1)].shape[0]/df_with_positions.shape[0]

0.08421052631578947

predicting a 2-1 home win is correct about 8.4% of the time

We next add some other features that are of some use

In [14]:
def preprocess_for_ml(df):
    """
    Preprocess football match data for machine learning with boosted decision trees.
    
    Parameters:
    df: DataFrame with match data including Date, Time, HomeTeam, AwayTeam, positions, and scores
    
    Returns:
    DataFrame with engineered features ready for ML
    """
    
    # Stadium coordinates (latitude, longitude) for all Premier League teams
    stadium_coords = {
        'Liverpool': (53.4308, -2.9608),
        'West Ham': (51.5386, -0.0164),
        'Bournemouth': (50.7352, -1.8382),
        'Burnley': (53.7889, -2.2302),
        'Crystal Palace': (51.3983, -0.0854),
        'Watford': (51.6499, -0.4016),
        'Tottenham': (51.6042, -0.0664),
        'Leicester': (52.6204, -1.1420),
        'Newcastle': (54.9756, -1.6217),
        'Man United': (53.4631, -2.2913),
        'Arsenal': (51.5549, -0.1084),
        'Aston Villa': (52.5092, -1.8848),
        'Brighton': (50.8614, -0.0831),
        'Everton': (53.4387, -2.9663),
        'Norwich': (52.6220, 1.3089),
        'Southampton': (50.9059, -1.3909),
        'Man City': (53.4831, -2.2004),
        'Sheffield United': (53.3702, -1.4708),
        'Chelsea': (51.4817, -0.1910),
        'Wolves': (52.5902, -2.1305),
        'Fulham': (51.4749, -0.2217),
        'West Brom': (52.5089, -1.9639),
        'Leeds': (53.7779, -1.5720),
        'Brentford': (51.4907, -0.2889),
        "Nott'm Forest": (52.9400, -1.1327),
        'Luton': (51.8844, -0.4318),
        'Ipswich': (52.0551, 1.1449),
        'Sunderland': (54.9144, -1.3882)
    }
    
    # COVID-19 attendance restrictions in Premier League
    # 0 = Normal attendance (full capacity)
    # 1 = Limited/reduced attendance (partial capacity)
    # 2 = Behind closed doors (no fans)
    
    # Key dates for Premier League COVID restrictions:
    NO_FANS_START = pd.Timestamp('2020-03-14')    # Last match day with fans was March 8-9, 2020
    NO_FANS_END = pd.Timestamp('2021-05-16')      # Last match day with no fans
    LIMITED_FANS_START = pd.Timestamp('2021-05-17')  # Limited fans started returning (2,000-10,000)
    LIMITED_FANS_END = pd.Timestamp('2021-08-13')    # Full capacity resumed for 2021-22 season start
    
    # Create a copy to avoid modifying original
    df_processed = df.copy()
    
    # Ensure Date is datetime
    if not pd.api.types.is_datetime64_any_dtype(df_processed['Date']):
        df_processed['Date'] = pd.to_datetime(df_processed['Date'])
    
    # Sort by date to ensure chronological order
    df_processed = df_processed.sort_values('Date').reset_index(drop=True)
    
    # ========================================================================
    # SEASON NUMBER (starting from 1 for the earliest season)
    # ========================================================================
    # If 'Season' column exists, use it; otherwise create it
    if 'Season' not in df_processed.columns:
        # Extract season from date (Aug-May football season)
        df_processed['Season'] = df_processed['Date'].apply(
            lambda x: f"{x.year}-{str(x.year+1)[-2:]}" if x.month >= 8 else f"{x.year-1}-{str(x.year)[-2:]}"
        )
    
    # Get unique seasons in chronological order
    unique_seasons = sorted(df_processed['Season'].unique())
    
    # Create season number mapping (1 for earliest, 2 for next, etc.)
    season_to_number = {season: idx + 1 for idx, season in enumerate(unique_seasons)}
    
    # Add Season_Number column
    df_processed['Season_Number'] = df_processed['Season'].map(season_to_number)
    
    print(f"Detected {len(unique_seasons)} seasons in dataset:")
    print(f"  Earliest season: {unique_seasons[0]} (Season_Number = 1)")
    print(f"  Latest season: {unique_seasons[-1]} (Season_Number = {len(unique_seasons)})")
    
    # ========================================================================
    # COVID-19 ATTENDANCE STATUS
    # ========================================================================
    def get_attendance_status(date):
        """
        Determine attendance status based on date.
        0 = Normal (full capacity)
        1 = Limited (reduced capacity)
        2 = Behind closed doors (no fans)
        """
        if date >= NO_FANS_START and date <= NO_FANS_END:
            return 2  # No fans
        elif date >= LIMITED_FANS_START and date <= LIMITED_FANS_END:
            return 1  # Limited fans
        else:
            return 0  # Normal attendance
    
    df_processed['Attendance_Status'] = df_processed['Date'].apply(get_attendance_status)
    
    # Print statistics
    attendance_counts = df_processed['Attendance_Status'].value_counts().sort_index()
    total_matches = len(df_processed)
    
    print("\n" + "="*70)
    print("COVID-19 ATTENDANCE STATUS")
    print("="*70)
    print(f"Total matches: {total_matches}")
    print(f"\nAttendance Status Distribution:")
    if 0 in attendance_counts.index:
        normal_count = attendance_counts[0]
        print(f"  0 (Normal - Full capacity): {normal_count} matches ({normal_count/total_matches*100:.1f}%)")
    if 1 in attendance_counts.index:
        limited_count = attendance_counts[1]
        print(f"  1 (Limited - Reduced capacity): {limited_count} matches ({limited_count/total_matches*100:.1f}%)")
        print(f"      Period: {LIMITED_FANS_START.date()} to {LIMITED_FANS_END.date()}")
    if 2 in attendance_counts.index:
        none_count = attendance_counts[2]
        print(f"  2 (Behind closed doors - No fans): {none_count} matches ({none_count/total_matches*100:.1f}%)")
        print(f"      Period: {NO_FANS_START.date()} to {NO_FANS_END.date()}")
    
    # ========================================================================
    # TEMPORAL FEATURES
    # ========================================================================
    df_processed['Year'] = df_processed['Date'].dt.year
    
    # Day of month (1-31) - sinusoidal encoding
    day_of_month = df_processed['Date'].dt.day
    df_processed['DayOfMonth_sin'] = np.sin(2 * np.pi * day_of_month / 31)
    df_processed['DayOfMonth_cos'] = np.cos(2 * np.pi * day_of_month / 31)
    
    # Month (1-12) - sinusoidal encoding
    month = df_processed['Date'].dt.month
    df_processed['Month_sin'] = np.sin(2 * np.pi * month / 12)
    df_processed['Month_cos'] = np.cos(2 * np.pi * month / 12)
    
    # Day of week (0-6, Monday=0) - sinusoidal encoding
    day_of_week = df_processed['Date'].dt.dayofweek
    df_processed['DayOfWeek_sin'] = np.sin(2 * np.pi * day_of_week / 7)
    df_processed['DayOfWeek_cos'] = np.cos(2 * np.pi * day_of_week / 7)
    
    # ========================================================================
    # TIME OF DAY
    # ========================================================================
    def time_to_float(time_str):
        if pd.isna(time_str):
            return np.nan
        try:
            hours, minutes = time_str.split(':')
            return int(hours) + int(minutes) / 60
        except:
            return np.nan
    
    df_processed['Time_float'] = df_processed['Time'].apply(time_to_float)
    
    # ========================================================================
    # STADIUM COORDINATES AND DISTANCE
    # ========================================================================
    df_processed['HomeTeam_Lat'] = df_processed['HomeTeam'].map(lambda x: stadium_coords.get(x, (np.nan, np.nan))[0])
    df_processed['HomeTeam_Lon'] = df_processed['HomeTeam'].map(lambda x: stadium_coords.get(x, (np.nan, np.nan))[1])
    df_processed['AwayTeam_Lat'] = df_processed['AwayTeam'].map(lambda x: stadium_coords.get(x, (np.nan, np.nan))[0])
    df_processed['AwayTeam_Lon'] = df_processed['AwayTeam'].map(lambda x: stadium_coords.get(x, (np.nan, np.nan))[1])
    
    # Calculate Euclidean distance between stadiums
    def calculate_distance(row):
        home_lat = row['HomeTeam_Lat']
        home_lon = row['HomeTeam_Lon']
        away_lat = row['AwayTeam_Lat']
        away_lon = row['AwayTeam_Lon']
        
        if pd.isna(home_lat) or pd.isna(away_lat):
            return np.nan
        
        distance = np.sqrt((home_lat - away_lat)**2 + (home_lon - away_lon)**2)
        return distance
    
    df_processed['Stadium_Distance'] = df_processed.apply(calculate_distance, axis=1)
    
    # ========================================================================
    # TEAM POSITIONS
    # ========================================================================
    df_processed['Position_Difference'] = df_processed['HomeTeam_Position'] - df_processed['AwayTeam_Position']
    
    # ========================================================================
    # SELECT FINAL COLUMNS
    # ========================================================================
    final_columns = [
        'DayOfMonth_sin', 'DayOfMonth_cos',
        'Month_sin', 'Month_cos',
        'DayOfWeek_sin', 'DayOfWeek_cos',
        'Year',
        'Season_Number',
        'Attendance_Status',
        'Time_float',
        'HomeTeam_Lat', 'HomeTeam_Lon',
        'AwayTeam_Lat', 'AwayTeam_Lon',
        'Stadium_Distance',
        'HomeTeam_Position', 'AwayTeam_Position',
        'Position_Difference',
        'FTHG', 'FTAG'
    ]
    
    df_final = df_processed[final_columns].copy()
    
    print(f"\nFinal preprocessed dataset shape: {df_final.shape}")
    print(f"Features created: {len(final_columns) - 2} (excluding FTHG and FTAG)")
    
    return df_final

In [15]:
# ============================================================================
# USAGE
# ============================================================================

# Preprocess the data
df_ml_ready = preprocess_for_ml(df_with_positions)

# Display info about new features
print("\n" + "="*70)
print("NEW FEATURES SUMMARY")
print("="*70)
print(f"\nSeason_Number range: {df_ml_ready['Season_Number'].min()} to {df_ml_ready['Season_Number'].max()}")
print(f"\nAttendance_Status distribution:")
print(df_ml_ready['Attendance_Status'].value_counts().sort_index())
print("\nSample of preprocessed data:")
#print(df_ml_ready.head(10))

Detected 7 seasons in dataset:
  Earliest season: 2019-20 (Season_Number = 1)
  Latest season: 2025-26 (Season_Number = 7)

COVID-19 ATTENDANCE STATUS
Total matches: 2660

Attendance Status Distribution:
  0 (Normal - Full capacity): 2187 matches (82.2%)
  1 (Limited - Reduced capacity): 21 matches (0.8%)
      Period: 2021-05-17 to 2021-08-13
  2 (Behind closed doors - No fans): 452 matches (17.0%)
      Period: 2020-03-14 to 2021-05-16

Final preprocessed dataset shape: (2660, 20)
Features created: 18 (excluding FTHG and FTAG)

NEW FEATURES SUMMARY

Season_Number range: 1 to 7

Attendance_Status distribution:
Attendance_Status
0    2187
1      21
2     452
Name: count, dtype: int64

Sample of preprocessed data:


In [23]:
df_ml_ready.head()

,DayOfMonth_sin,DayOfMonth_cos,Month_sin,Month_cos,DayOfWeek_sin,DayOfWeek_cos,Year,Season_Number,Attendance_Status,Time_float,HomeTeam_Lat,HomeTeam_Lon,AwayTeam_Lat,AwayTeam_Lon,Stadium_Distance,HomeTeam_Position,AwayTeam_Position,Position_Difference,FTHG,FTAG
0,0.968077,-0.250653,-0.866025,-0.5,-0.433884,-0.900969,2019,1,0,20.0,53.4308,-2.9608,52.6220,1.3089,4.345629,5.0,17.0,-12.0,4,1
1,0.897805,-0.440394,-0.866025,-0.5,-0.974928,-0.222521,2019,1,0,12.5,51.5386,-0.0164,53.4831,-2.2004,2.924198,12.0,3.0,9.0,0,5
2,0.897805,-0.440394,-0.866025,-0.5,-0.974928,-0.222521,2019,1,0,15.0,50.7352,-1.8382,53.3702,-1.4708,2.660490,6.0,17.0,-11.0,1,1
3,0.897805,-0.440394,-0.866025,-0.5,-0.974928,-0.222521,2019,1,0,15.0,53.7889,-2.2302,50.9059,-1.3909,3.002684,13.0,16.0,-3.0,3,0
4,0.897805,-0.440394,-0.866025,-0.5,-0.974928,-0.222521,2019,1,0,15.0,51.3983,-0.0854,53.4387,-2.9663,3.530272,12.0,11.0,1.0,0,0


## The models themselves

The values used are those obtained following a grid-search to fine-tune the model.

In [16]:
# ============================================================================
# HYPERPARAMETERS - Adjust these for tuning
# ============================================================================
N_FOLDS = 10
RANDOM_STATE = 42

# Early stopping parameters (shared)
EARLY_STOPPING_ROUNDS = 5
EVAL_METRIC = 'mae'

# HOME GOALS MODEL HYPERPARAMETERS
HOME_MAX_DEPTH = 4
HOME_LEARNING_RATE = 0.15
HOME_N_ESTIMATORS = 1_000
HOME_MIN_CHILD_WEIGHT = 1
HOME_SUBSAMPLE = 0.8
HOME_COLSAMPLE_BYTREE = 0.8
HOME_GAMMA = 0
HOME_REG_ALPHA = 0  # L1 regularization
HOME_REG_LAMBDA = 1  # L2 regularization

# AWAY GOALS MODEL HYPERPARAMETERS
AWAY_MAX_DEPTH = 6
AWAY_LEARNING_RATE = 0.15
AWAY_N_ESTIMATORS = 1_000
AWAY_MIN_CHILD_WEIGHT = 1
AWAY_SUBSAMPLE = 0.8
AWAY_COLSAMPLE_BYTREE = 0.8
AWAY_GAMMA = 0
AWAY_REG_ALPHA = 0  # L1 regularization
AWAY_REG_LAMBDA = 1  # L2 regularization

# ============================================================================
# MODEL TRAINING AND EVALUATION
# ============================================================================

def train_and_evaluate_models(df_ml_ready):
    """
    Train XGBoost models to predict home and away goals using k-fold cross-validation.
    
    Parameters:
    df_ml_ready: Preprocessed dataframe with features and targets
    
    Returns:
    Dictionary with evaluation metrics and predictions
    """
    
    # Separate features and targets
    feature_cols = [col for col in df_ml_ready.columns if col not in ['FTHG', 'FTAG']]
    X = df_ml_ready[feature_cols].copy()
    y_home = df_ml_ready['FTHG'].copy()
    y_away = df_ml_ready['FTAG'].copy()
    
    # Remove rows with missing values
    valid_mask = ~(X.isna().any(axis=1) | y_home.isna() | y_away.isna())
    X = X[valid_mask].reset_index(drop=True)
    y_home = y_home[valid_mask].reset_index(drop=True)
    y_away = y_away[valid_mask].reset_index(drop=True)
    
    print(f"Training on {len(X)} matches with {len(feature_cols)} features")
    print(f"Features: {feature_cols}\n")
    
    # Initialize k-fold cross-validation
    kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    
    # Storage for results
    results = {
        'home_mae': [],
        'away_mae': [],
        'combined_mae': [],
        'perfect_predictions': [],
        'home_predictions': np.zeros(len(X)),
        'away_predictions': np.zeros(len(X)),
        'fold_details': []
    }
    
    # XGBoost parameters for HOME model
    xgb_params_home = {
        'max_depth': HOME_MAX_DEPTH,
        'learning_rate': HOME_LEARNING_RATE,
        'n_estimators': HOME_N_ESTIMATORS,
        'min_child_weight': HOME_MIN_CHILD_WEIGHT,
        'subsample': HOME_SUBSAMPLE,
        'colsample_bytree': HOME_COLSAMPLE_BYTREE,
        'gamma': HOME_GAMMA,
        'reg_alpha': HOME_REG_ALPHA,
        'reg_lambda': HOME_REG_LAMBDA,
        'objective': 'count:poisson',  # Poisson loss for count data
        'random_state': RANDOM_STATE,
        'n_jobs': -1,
        'early_stopping_rounds': EARLY_STOPPING_ROUNDS  # Add here
    }
    
    # XGBoost parameters for AWAY model
    xgb_params_away = {
        'max_depth': AWAY_MAX_DEPTH,
        'learning_rate': AWAY_LEARNING_RATE,
        'n_estimators': AWAY_N_ESTIMATORS,
        'min_child_weight': AWAY_MIN_CHILD_WEIGHT,
        'subsample': AWAY_SUBSAMPLE,
        'colsample_bytree': AWAY_COLSAMPLE_BYTREE,
        'gamma': AWAY_GAMMA,
        'reg_alpha': AWAY_REG_ALPHA,
        'reg_lambda': AWAY_REG_LAMBDA,
        'objective': 'count:poisson',  # Poisson loss for count data
        'random_state': RANDOM_STATE,
        'n_jobs': -1,
        'early_stopping_rounds': EARLY_STOPPING_ROUNDS  # Add here
    }
    
    print(f"Starting {N_FOLDS}-fold cross-validation...\n")
    print("="*70)
    print("HOME MODEL HYPERPARAMETERS:")
    for key, value in xgb_params_home.items():
        if key not in ['objective', 'random_state', 'n_jobs']:
            print(f"  {key}: {value}")
    print("\nAWAY MODEL HYPERPARAMETERS:")
    for key, value in xgb_params_away.items():
        if key not in ['objective', 'random_state', 'n_jobs']:
            print(f"  {key}: {value}")
    print("="*70)
    
    # Perform k-fold cross-validation
    for fold, (train_idx, val_idx) in enumerate(kf.split(X), 1):
        print(f"\nFOLD {fold}/{N_FOLDS}")
        print("-"*70)
        
        # Split data
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_home_train, y_home_val = y_home.iloc[train_idx], y_home.iloc[val_idx]
        y_away_train, y_away_val = y_away.iloc[train_idx], y_away.iloc[val_idx]
        
        # Train home goals model
        print("Training Home Goals model...")
        model_home = xgb.XGBRegressor(**xgb_params_home)
        model_home.fit(
            X_train, y_home_train,
            eval_set=[(X_val, y_home_val)],
            verbose=False
        )
        
        # Train away goals model
        print("Training Away Goals model...")
        model_away = xgb.XGBRegressor(**xgb_params_away)
        model_away.fit(
            X_train, y_away_train,
            eval_set=[(X_val, y_away_val)],
            verbose=False
        )
        
        # Make predictions on validation set
        pred_home_raw = model_home.predict(X_val)
        pred_away_raw = model_away.predict(X_val)
        
        # Round to nearest integer (like real match scores)
        pred_home = np.round(pred_home_raw).astype(int)
        pred_away = np.round(pred_away_raw).astype(int)
        
        # Store predictions for later analysis
        results['home_predictions'][val_idx] = pred_home
        results['away_predictions'][val_idx] = pred_away
        
        # Calculate metrics
        mae_home = mean_absolute_error(y_home_val, pred_home)
        mae_away = mean_absolute_error(y_away_val, pred_away)
        mae_combined = (mae_home + mae_away) / 2
        
        # Calculate percentage of perfect predictions (both scores correct)
        perfect_matches = ((pred_home == y_home_val) & (pred_away == y_away_val)).sum()
        perfect_percentage = (perfect_matches / len(y_home_val)) * 100
        
        # Store results
        results['home_mae'].append(mae_home)
        results['away_mae'].append(mae_away)
        results['combined_mae'].append(mae_combined)
        results['perfect_predictions'].append(perfect_percentage)
        
        fold_detail = {
            'fold': fold,
            'model_home': model_home,
            'model_away': model_away,
            'best_iteration_home': model_home.best_iteration,
            'best_iteration_away': model_away.best_iteration
        }
        results['fold_details'].append(fold_detail)
        
        # Print fold results
        print(f"\nFold {fold} Results:")
        print(f"  Home Goals MAE: {mae_home:.4f}")
        print(f"  Away Goals MAE: {mae_away:.4f}")
        print(f"  Combined MAE: {mae_combined:.4f}")
        print(f"  Perfect Predictions: {perfect_matches}/{len(y_home_val)} ({perfect_percentage:.2f}%)")
        print(f"  Best iterations - Home: {model_home.best_iteration}, Away: {model_away.best_iteration}")
    
    # Calculate overall statistics
    print("\n" + "="*70)
    print("OVERALL CROSS-VALIDATION RESULTS")
    print("="*70)
    print(f"\nHome Goals MAE: {np.mean(results['home_mae']):.4f} ± {np.std(results['home_mae']):.4f}")
    print(f"Away Goals MAE: {np.mean(results['away_mae']):.4f} ± {np.std(results['away_mae']):.4f}")
    print(f"Combined MAE: {np.mean(results['combined_mae']):.4f} ± {np.std(results['combined_mae']):.4f}")
    print(f"Perfect Predictions: {np.mean(results['perfect_predictions']):.2f}% ± {np.std(results['perfect_predictions']):.2f}%")
    
    # Overall perfect predictions on all data
    pred_home_all = np.round(results['home_predictions']).astype(int)
    pred_away_all = np.round(results['away_predictions']).astype(int)
    perfect_all = ((pred_home_all == y_home) & (pred_away_all == y_away)).sum()
    perfect_pct_all = (perfect_all / len(y_home)) * 100
    
    print(f"\nOverall Perfect Predictions (all folds): {perfect_all}/{len(y_home)} ({perfect_pct_all:.2f}%)")
    
    # Store final summary
    results['summary'] = {
        'mean_home_mae': np.mean(results['home_mae']),
        'std_home_mae': np.std(results['home_mae']),
        'mean_away_mae': np.mean(results['away_mae']),
        'std_away_mae': np.std(results['away_mae']),
        'mean_combined_mae': np.mean(results['combined_mae']),
        'std_combined_mae': np.std(results['combined_mae']),
        'mean_perfect_pct': np.mean(results['perfect_predictions']),
        'std_perfect_pct': np.std(results['perfect_predictions']),
        'overall_perfect_count': perfect_all,
        'overall_perfect_pct': perfect_pct_all,
        'total_samples': len(y_home)
    }
    
    return results, X, y_home, y_away


def train_final_models(X, y_home, y_away):
    """
    Train final models on all available data.
    
    Parameters:
    X: Feature matrix
    y_home: Home goals target
    y_away: Away goals target
    
    Returns:
    Tuple of (home_model, away_model)
    """
    
    # XGBoost parameters for HOME model
    xgb_params_home = {
        'max_depth': HOME_MAX_DEPTH,
        'learning_rate': HOME_LEARNING_RATE,
        'n_estimators': HOME_N_ESTIMATORS,
        'min_child_weight': HOME_MIN_CHILD_WEIGHT,
        'subsample': HOME_SUBSAMPLE,
        'colsample_bytree': HOME_COLSAMPLE_BYTREE,
        'gamma': HOME_GAMMA,
        'reg_alpha': HOME_REG_ALPHA,
        'reg_lambda': HOME_REG_LAMBDA,
        'objective': 'count:poisson',
        'random_state': RANDOM_STATE,
        'n_jobs': -1,
        'early_stopping_rounds': EARLY_STOPPING_ROUNDS
    }
    
    # XGBoost parameters for AWAY model
    xgb_params_away = {
        'max_depth': AWAY_MAX_DEPTH,
        'learning_rate': AWAY_LEARNING_RATE,
        'n_estimators': AWAY_N_ESTIMATORS,
        'min_child_weight': AWAY_MIN_CHILD_WEIGHT,
        'subsample': AWAY_SUBSAMPLE,
        'colsample_bytree': AWAY_COLSAMPLE_BYTREE,
        'gamma': AWAY_GAMMA,
        'reg_alpha': AWAY_REG_ALPHA,
        'reg_lambda': AWAY_REG_LAMBDA,
        'objective': 'count:poisson',
        'random_state': RANDOM_STATE,
        'n_jobs': -1,
        'early_stopping_rounds': EARLY_STOPPING_ROUNDS
    }
    
    print("\n" + "="*70)
    print("TRAINING FINAL MODELS ON ALL DATA")
    print("="*70)
    
    # Split data for early stopping (90-10 split)
    split_idx = int(0.9 * len(X))
    X_train, X_val = X.iloc[:split_idx], X.iloc[split_idx:]
    y_home_train, y_home_val = y_home.iloc[:split_idx], y_home.iloc[split_idx:]
    y_away_train, y_away_val = y_away.iloc[:split_idx], y_away.iloc[split_idx:]
    
    # Train final home model
    print("\nTraining final Home Goals model...")
    final_model_home = xgb.XGBRegressor(**xgb_params_home)
    final_model_home.fit(
        X_train, y_home_train,
        eval_set=[(X_val, y_home_val)],
        verbose=False
    )
    print(f"Best iteration: {final_model_home.best_iteration}")
    
    # Train final away model
    print("Training final Away Goals model...")
    final_model_away = xgb.XGBRegressor(**xgb_params_away)
    final_model_away.fit(
        X_train, y_away_train,
        eval_set=[(X_val, y_away_val)],
        verbose=False
    )
    print(f"Best iteration: {final_model_away.best_iteration}")
    
    return final_model_home, final_model_away

In [20]:
# ============================================================================
# USAGE EXAMPLE
# ============================================================================

# Train and evaluate with cross-validation
results, X, y_home, y_away = train_and_evaluate_models(df_ml_ready)

# Train final models on all data
final_home_model, final_away_model = train_final_models(X, y_home, y_away)

Training on 2660 matches with 18 features
Features: ['DayOfMonth_sin', 'DayOfMonth_cos', 'Month_sin', 'Month_cos', 'DayOfWeek_sin', 'DayOfWeek_cos', 'Year', 'Season_Number', 'Attendance_Status', 'Time_float', 'HomeTeam_Lat', 'HomeTeam_Lon', 'AwayTeam_Lat', 'AwayTeam_Lon', 'Stadium_Distance', 'HomeTeam_Position', 'AwayTeam_Position', 'Position_Difference']

Starting 10-fold cross-validation...

HOME MODEL HYPERPARAMETERS:
  max_depth: 4
  learning_rate: 0.15
  n_estimators: 1000
  min_child_weight: 1
  subsample: 0.8
  colsample_bytree: 0.8
  gamma: 0
  reg_alpha: 0
  reg_lambda: 1
  early_stopping_rounds: 5

AWAY MODEL HYPERPARAMETERS:
  max_depth: 6
  learning_rate: 0.15
  n_estimators: 1000
  min_child_weight: 1
  subsample: 0.8
  colsample_bytree: 0.8
  gamma: 0
  reg_alpha: 0
  reg_lambda: 1
  early_stopping_rounds: 5

FOLD 1/10
----------------------------------------------------------------------
Training Home Goals model...
Training Away Goals model...

Fold 1 Results:
  Home Go

## Predictions

The below illustrates how to make actual predictionsusing the above

In [18]:
# ============================================================================
# PREDICT A SPECIFIC MATCH
# ============================================================================

def predict_match(home_team, away_team, match_date, kick_off_time, 
                  final_home_model, final_away_model, 
                  df_with_positions, final_standings):
    """
    Predict the score for a specific match.
    
    Parameters:
    home_team: Name of home team (e.g., 'Man United')
    away_team: Name of away team (e.g., 'Man City')
    match_date: Date string 'YYYY-MM-DD' or datetime object
    kick_off_time: Time string 'HH:MM' (e.g., '12:30')
    final_home_model: Trained XGBoost model for home goals
    final_away_model: Trained XGBoost model for away goals
    df_with_positions: Historical data with positions
    final_standings: List of current league standings
    
    Returns:
    Dictionary with prediction details
    """
    
    print("="*70)
    print("MATCH PREDICTION")
    print("="*70)
    print(f"\n{home_team} vs {away_team}")
    print(f"Date: {match_date}")
    print(f"Kick-off: {kick_off_time}")
    
    # Convert to datetime if needed
    if isinstance(match_date, str):
        match_date = pd.to_datetime(match_date)
    
    # Stadium coordinates (same as in preprocessing)
    stadium_coords = {
        'Liverpool': (53.4308, -2.9608),
        'West Ham': (51.5386, -0.0164),
        'Bournemouth': (50.7352, -1.8382),
        'Burnley': (53.7889, -2.2302),
        'Crystal Palace': (51.3983, -0.0854),
        'Watford': (51.6499, -0.4016),
        'Tottenham': (51.6042, -0.0664),
        'Leicester': (52.6204, -1.1420),
        'Newcastle': (54.9756, -1.6217),
        'Man United': (53.4631, -2.2913),
        'Arsenal': (51.5549, -0.1084),
        'Aston Villa': (52.5092, -1.8848),
        'Brighton': (50.8614, -0.0831),
        'Brighton & Hove Albion': (50.8614, -0.0831),
        'Everton': (53.4387, -2.9663),
        'Norwich': (52.6220, 1.3089),
        'Southampton': (50.9059, -1.3909),
        'Man City': (53.4831, -2.2004),
        'Sheffield United': (53.3702, -1.4708),
        'Chelsea': (51.4817, -0.1910),
        'Wolves': (52.5902, -2.1305),
        'Fulham': (51.4749, -0.2217),
        'West Brom': (52.5089, -1.9639),
        'Leeds': (53.7779, -1.5720),
        'Brentford': (51.4907, -0.2889),
        "Nott'm Forest": (52.9400, -1.1327),
        'Luton': (51.8844, -0.4318),
        'Ipswich': (52.0551, 1.1449),
        'Sunderland': (54.9144, -1.3882)
    }
    
    # COVID attendance dates
    NO_FANS_START = pd.Timestamp('2020-03-14')
    NO_FANS_END = pd.Timestamp('2021-05-16')
    LIMITED_FANS_START = pd.Timestamp('2021-05-17')
    LIMITED_FANS_END = pd.Timestamp('2021-08-13')
    
    # ========================================================================
    # TEMPORAL FEATURES
    # ========================================================================
    
    # Day of month
    day_of_month = match_date.day
    day_of_month_sin = np.sin(2 * np.pi * day_of_month / 31)
    day_of_month_cos = np.cos(2 * np.pi * day_of_month / 31)
    
    # Month
    month = match_date.month
    month_sin = np.sin(2 * np.pi * month / 12)
    month_cos = np.cos(2 * np.pi * month / 12)
    
    # Day of week
    day_of_week = match_date.dayofweek
    day_of_week_sin = np.sin(2 * np.pi * day_of_week / 7)
    day_of_week_cos = np.cos(2 * np.pi * day_of_week / 7)
    
    # Year
    year = match_date.year
    
    # Season number (detect from historical data)
    if 'Season' in df_with_positions.columns:
        unique_seasons = sorted(df_with_positions['Season'].unique())
        # Determine current season from date
        if match_date.month >= 8:
            current_season = f"{match_date.year}-{str(match_date.year+1)[-2:]}"
        else:
            current_season = f"{match_date.year-1}-{str(match_date.year)[-2:]}"
        
        if current_season in unique_seasons:
            season_number = unique_seasons.index(current_season) + 1
        else:
            # New season not in historical data
            season_number = len(unique_seasons) + 1
    else:
        season_number = 1
    
    # Attendance status
    if match_date >= NO_FANS_START and match_date <= NO_FANS_END:
        attendance_status = 2  # No fans
    elif match_date >= LIMITED_FANS_START and match_date <= LIMITED_FANS_END:
        attendance_status = 1  # Limited fans
    else:
        attendance_status = 0  # Normal
    
    # Time as float
    hours, minutes = kick_off_time.split(':')
    time_float = int(hours) + int(minutes) / 60
    
    # ========================================================================
    # STADIUM COORDINATES AND DISTANCE
    # ========================================================================
    
    home_lat, home_lon = stadium_coords.get(home_team, (np.nan, np.nan))
    away_lat, away_lon = stadium_coords.get(away_team, (np.nan, np.nan))
    
    stadium_distance = np.sqrt((home_lat - away_lat)**2 + (home_lon - away_lon)**2)
    
    # ========================================================================
    # TEAM POSITIONS
    # ========================================================================
    
    try:
        home_position = final_standings.index(home_team) + 1
    except ValueError:
        print(f"\n⚠ Warning: {home_team} not found in current standings")
        home_position = 10  # Default to mid-table
    
    try:
        away_position = final_standings.index(away_team) + 1
    except ValueError:
        print(f"\n⚠ Warning: {away_team} not found in current standings")
        away_position = 10  # Default to mid-table
    
    position_difference = home_position - away_position
    
    # ========================================================================
    # CREATE FEATURE DATAFRAME
    # ========================================================================
    
    match_features = pd.DataFrame({
        'DayOfMonth_sin': [day_of_month_sin],
        'DayOfMonth_cos': [day_of_month_cos],
        'Month_sin': [month_sin],
        'Month_cos': [month_cos],
        'DayOfWeek_sin': [day_of_week_sin],
        'DayOfWeek_cos': [day_of_week_cos],
        'Year': [year],
        'Season_Number': [season_number],
        'Attendance_Status': [attendance_status],
        'Time_float': [time_float],
        'HomeTeam_Lat': [home_lat],
        'HomeTeam_Lon': [home_lon],
        'AwayTeam_Lat': [away_lat],
        'AwayTeam_Lon': [away_lon],
        'Stadium_Distance': [stadium_distance],
        'HomeTeam_Position': [home_position],
        'AwayTeam_Position': [away_position],
        'Position_Difference': [position_difference]
    })
    
    # Print feature values for transparency
    print("\n" + "-"*70)
    print("FEATURE VALUES")
    print("-"*70)
    print(f"Date features:")
    print(f"  Day: {match_date.day} → sin={day_of_month_sin:.3f}, cos={day_of_month_cos:.3f}")
    print(f"  Month: {match_date.month} ({match_date.strftime('%B')}) → sin={month_sin:.3f}, cos={month_cos:.3f}")
    print(f"  Day of week: {match_date.strftime('%A')} → sin={day_of_week_sin:.3f}, cos={day_of_week_cos:.3f}")
    print(f"  Year: {year}")
    print(f"  Season number: {season_number}")
    print(f"\nTime:")
    print(f"  Kick-off: {kick_off_time} → {time_float:.2f}")
    print(f"\nAttendance:")
    print(f"  Status: {attendance_status} ({'Normal' if attendance_status==0 else 'Limited' if attendance_status==1 else 'No fans'})")
    print(f"\nLocation:")
    print(f"  {home_team} stadium: ({home_lat:.4f}, {home_lon:.4f})")
    print(f"  {away_team} stadium: ({away_lat:.4f}, {away_lon:.4f})")
    print(f"  Distance: {stadium_distance:.4f}")
    print(f"\nLeague positions:")
    print(f"  {home_team}: {home_position} / {len(final_standings)}")
    print(f"  {away_team}: {away_position} / {len(final_standings)}")
    print(f"  Position difference: {position_difference:+d} ({'home advantage' if position_difference < 0 else 'away advantage' if position_difference > 0 else 'equal'})")
    
    # ========================================================================
    # MAKE PREDICTIONS
    # ========================================================================
    
    print("\n" + "="*70)
    print("PREDICTIONS")
    print("="*70)
    
    # Get raw predictions
    home_goals_raw = final_home_model.predict(match_features)[0]
    away_goals_raw = final_away_model.predict(match_features)[0]
    
    # Round to integers
    home_goals_pred = int(np.round(home_goals_raw))
    away_goals_pred = int(np.round(away_goals_raw))
    
    print(f"\n{home_team} (Home): {home_goals_pred} goals")
    print(f"  (raw prediction: {home_goals_raw:.2f})")
    print(f"\n{away_team} (Away): {away_goals_pred} goals")
    print(f"  (raw prediction: {away_goals_raw:.2f})")
    
    # Determine result
    if home_goals_pred > away_goals_pred:
        result = f"{home_team} WIN"
        result_code = 'H'
    elif away_goals_pred > home_goals_pred:
        result = f"{away_team} WIN"
        result_code = 'A'
    else:
        result = "DRAW"
        result_code = 'D'
    
    print(f"\n" + "="*70)
    print(f"PREDICTED SCORE: {home_team} {home_goals_pred} - {away_goals_pred} {away_team}")
    print(f"PREDICTED RESULT: {result}")
    print("="*70)
    
    # ========================================================================
    # PREDICTION INTERVALS (OPTIONAL)
    # ========================================================================
    
    # Since we're using point predictions, we can give a sense of uncertainty
    # based on the raw predictions
    
    print("\n" + "-"*70)
    print("PREDICTION CONFIDENCE")
    print("-"*70)
    
    # If raw prediction is close to 0.5 boundary, less confident
    home_confidence = min(abs(home_goals_raw - np.floor(home_goals_raw)), 
                         abs(home_goals_raw - np.ceil(home_goals_raw)))
    away_confidence = min(abs(away_goals_raw - np.floor(away_goals_raw)), 
                         abs(away_goals_raw - np.ceil(away_goals_raw)))
    
    # Scale to 0-100%
    home_confidence_pct = (1 - 2 * home_confidence) * 100
    away_confidence_pct = (1 - 2 * away_confidence) * 100
    
    print(f"{home_team} goals confidence: {home_confidence_pct:.1f}%")
    print(f"{away_team} goals confidence: {away_confidence_pct:.1f}%")
    
    if home_confidence_pct < 50 or away_confidence_pct < 50:
        print("\n⚠ Note: Low confidence suggests prediction is uncertain")
        print(f"   Consider {home_team} could score {int(np.floor(home_goals_raw))} or {int(np.ceil(home_goals_raw))} goals")
        print(f"   Consider {away_team} could score {int(np.floor(away_goals_raw))} or {int(np.ceil(away_goals_raw))} goals")
    
    # ========================================================================
    # RETURN RESULTS
    # ========================================================================
    
    return {
        'home_team': home_team,
        'away_team': away_team,
        'date': match_date,
        'kick_off': kick_off_time,
        'predicted_home_goals': home_goals_pred,
        'predicted_away_goals': away_goals_pred,
        'predicted_result': result_code,
        'raw_home_goals': home_goals_raw,
        'raw_away_goals': away_goals_raw,
        'home_position': home_position,
        'away_position': away_position,
        'features': match_features
    }


# ============================================================================
# PREDICT MULTIPLE MATCHES
# ============================================================================

def predict_multiple_matches(matches_list, final_home_model, final_away_model, 
                             df_with_positions, final_standings):
    """
    Predict multiple matches at once.
    
    Parameters:
    matches_list: List of dictionaries with keys: home_team, away_team, date, time
    
    Returns:
    DataFrame with all predictions
    """
    
    predictions = []
    
    for match in matches_list:
        pred = predict_match(
            match['home_team'],
            match['away_team'],
            match['date'],
            match['time'],
            final_home_model,
            final_away_model,
            df_with_positions,
            final_standings
        )
        predictions.append(pred)
        print("\n")
    
    # Create summary DataFrame
    results_df = pd.DataFrame([
        {
            'Match': f"{p['home_team']} vs {p['away_team']}",
            'Date': p['date'].strftime('%Y-%m-%d'),
            'Time': p['kick_off'],
            'Home': p['home_team'],
            'Away': p['away_team'],
            'Predicted_Score': f"{p['predicted_home_goals']}-{p['predicted_away_goals']}",
            'Result': p['predicted_result'],
            'Home_Goals': p['predicted_home_goals'],
            'Away_Goals': p['predicted_away_goals']
        }
        for p in predictions
    ])
    
    print("="*70)
    print("PREDICTIONS SUMMARY")
    print("="*70)
    print(results_df.to_string(index=False))
    
    return results_df, predictions

In [21]:
# ============================================================================
# USAGE EXAMPLES
# ============================================================================

# Example 1: Single match prediction
prediction = predict_match(
    home_team='Man United',
    away_team='Man City',
    match_date='2026-01-17',
    kick_off_time='12:30',
    final_home_model=final_home_model,
    final_away_model=final_away_model,
    df_with_positions=df_with_positions,
    final_standings=final_standings
)

# Example 2: Predict multiple matches
upcoming_matches = [
    {'home_team': 'Man United', 'away_team': 'Man City', 'date': '2026-01-17', 'time': '12:30'},
    {'home_team': 'Liverpool', 'away_team': 'Arsenal', 'date': '2026-01-18', 'time': '16:30'},
    {'home_team': 'Chelsea', 'away_team': 'Tottenham', 'date': '2026-01-19', 'time': '14:00'}
]

predictions_df, all_predictions = predict_multiple_matches(
    upcoming_matches,
    final_home_model,
    final_away_model,
    df_with_positions,
    final_standings
)

# Save predictions to CSV
predictions_df.to_csv('match_predictions.csv', index=False)
print("\n✓ Predictions saved to 'match_predictions.csv'")

MATCH PREDICTION

Man United vs Man City
Date: 2026-01-17
Kick-off: 12:30

----------------------------------------------------------------------
FEATURE VALUES
----------------------------------------------------------------------
Date features:
  Day: 17 → sin=-0.299, cos=-0.954
  Month: 1 (January) → sin=0.500, cos=0.866
  Day of week: Saturday → sin=-0.975, cos=-0.223
  Year: 2026
  Season number: 7

Time:
  Kick-off: 12:30 → 12.50

Attendance:
  Status: 0 (Normal)

Location:
  Man United stadium: (53.4631, -2.2913)
  Man City stadium: (53.4831, -2.2004)
  Distance: 0.0931

League positions:
  Man United: 16 / 20
  Man City: 15 / 20
  Position difference: +1 (away advantage)

PREDICTIONS

Man United (Home): 2 goals
  (raw prediction: 1.65)

Man City (Away): 1 goals
  (raw prediction: 1.24)

PREDICTED SCORE: Man United 2 - 1 Man City
PREDICTED RESULT: Man United WIN

----------------------------------------------------------------------
PREDICTION CONFIDENCE
------------------------